# Gatekeeper · 01 — Understand the data and split it by time

**Goal tonight:** load the IEEE-CIS data, see how rare fraud is and how it changes over time,
check for columns that leak the answer, and save three time-ordered splits:

| Split | Share | Used for |
|---|---|---|
| `train` | first ~70% | teaching the models |
| `valid` | next ~15% | tuning and choosing the champion |
| `replay` | last ~15% | the "live" stream the simulator sends later — **never trained on** |

**Run on Kaggle:** *Add Input → Competitions → IEEE-CIS Fraud Detection*, then *Run All*.

In [ ]:
import os, gc, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# Where the data lives. On Kaggle it's attached under /kaggle/input.
DATA_DIR = os.environ.get("GK_DATA_DIR", "/kaggle/input/ieee-fraud-detection")
OUT_DIR = os.environ.get("GK_OUT_DIR", "/kaggle/working")
os.makedirs(OUT_DIR, exist_ok=True)
print(os.listdir(DATA_DIR))

## 1. Load and join

`train_transaction` holds the payment itself; `train_identity` holds device and browser details
for only some payments. We join on `TransactionID` and shrink number types to save memory.
(The competition's `test_*` files have no fraud labels, so we only use `train_*`.)

In [ ]:
def shrink(df):
    for c in df.select_dtypes("float64").columns:
        df[c] = df[c].astype("float32")
    for c in df.select_dtypes("int64").columns:
        if c not in ("TransactionID", "TransactionDT"):
            df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

t0 = time.time()
tx = shrink(pd.read_csv(f"{DATA_DIR}/train_transaction.csv"))
idn = shrink(pd.read_csv(f"{DATA_DIR}/train_identity.csv"))
df = tx.merge(idn, on="TransactionID", how="left")
del tx, idn; gc.collect()

print(f"Loaded in {time.time() - t0:.0f}s")
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

## 2. How rare is fraud?

This number shapes everything later: if fraud is ~3.5%, a model that always says "genuine"
is already 96.5% "accurate", which is why we will **not** judge models on accuracy.

In [ ]:
rate = df["isFraud"].mean()
print(f"Fraud rate: {rate:.2%}  ({df['isFraud'].sum():,} of {len(df):,})")
print(f"Accuracy of a model that never says fraud: {1 - rate:.2%}")

## 3. Time

`TransactionDT` is seconds from an unknown start date, not a real timestamp.
We turn it into a day number and look at fraud over time — if the rate moves, a random split
would let models "see the future", so we split by time instead.

In [ ]:
df = df.sort_values("TransactionDT").reset_index(drop=True)
df["day"] = (df["TransactionDT"] // 86400).astype(int)
df["week"] = df["day"] // 7

weekly = df.groupby("week").agg(payments=("isFraud", "size"), fraud_rate=("isFraud", "mean"))
print(f"Data covers {df['day'].min()}–{df['day'].max()} days (~{df['day'].max() / 30:.1f} months)")

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(weekly.index, weekly["payments"], color="#cbd5e1", label="payments per week")
ax1.set_xlabel("week"); ax1.set_ylabel("payments")
ax2 = ax1.twinx()
ax2.plot(weekly.index, weekly["fraud_rate"] * 100, color="#dc2626", label="fraud rate %")
ax2.set_ylabel("fraud rate (%)")
fig.legend(loc="upper right"); ax1.set_title("Payments and fraud rate by week")
plt.tight_layout(); plt.show()

## 4. Missing values

Many columns are mostly empty (identity details exist for only some payments).
Tree models can cope with gaps, but columns that are almost always empty add little.

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
print(f"Columns >90% empty: {(missing > 0.9).sum()}")
print(f"Columns >50% empty: {(missing > 0.5).sum()}")
print(f"Columns with no gaps: {(missing == 0).sum()}")
missing.head(15).to_frame("share_missing")

## 5. Split by time: train → valid → replay

Cut points are placed on whole days so one day never sits in two splits.

In [ ]:
days = df["day"]
cut_train = days.quantile(0.70)
cut_valid = days.quantile(0.85)

split = np.where(days <= cut_train, "train", np.where(days <= cut_valid, "valid", "replay"))
df["split"] = split

summary = df.groupby("split", sort=False).agg(
    rows=("isFraud", "size"), fraud_rate=("isFraud", "mean"),
    first_day=("day", "min"), last_day=("day", "max"))
summary = summary.loc[["train", "valid", "replay"]]
summary["share"] = summary["rows"] / len(df)
summary

## 6. Leakage check

A column "leaks" when it gives away the answer in a way the live system couldn't know.
Quick test: how well does **each column on its own** separate fraud from genuine (AUC; 0.5 = useless,
1.0 = perfect)? Anything near 1.0 by itself deserves a hard look before we trust it.
We score on a sample of the training split to keep this fast.

In [ ]:
sample = df[df["split"] == "train"].sample(n=min(100_000, (df["split"] == "train").sum()), random_state=42)
y = sample["isFraud"]
skip = {"isFraud", "TransactionID", "split", "day", "week"}

rows = []
for c in sample.columns:
    if c in skip or not pd.api.types.is_numeric_dtype(sample[c]):
        continue
    x = sample[c].fillna(sample[c].median() if sample[c].notna().any() else 0)
    if x.nunique() < 2:
        continue
    auc = roc_auc_score(y, x)
    rows.append((c, max(auc, 1 - auc), sample[c].isna().mean()))

single = pd.DataFrame(rows, columns=["column", "auc_alone", "share_missing"]).sort_values("auc_alone", ascending=False)
print("Columns that separate fraud best on their own:")
single.head(15)

In [ ]:
suspicious = single[single["auc_alone"] > 0.90]
if suspicious.empty:
    print("No single column separates fraud almost perfectly — no obvious leak.")
else:
    print("Look closely at these before using them:")
    print(suspicious.to_string(index=False))

# Known trap in this dataset: TransactionID and TransactionDT just count up over time.
# They are kept for ordering and replay, but must NOT be fed to the models as features.
print("\nNever use as model features: TransactionID, TransactionDT, day, week, split")

## 7. Save the splits

Saved as Parquet (much smaller and faster than CSV). Download them from the notebook's
**Output** panel, or attach this notebook's output as input to tomorrow's notebook.

In [ ]:
for name in ["train", "valid", "replay"]:
    part = df[df["split"] == name].drop(columns=["split"])
    path = f"{OUT_DIR}/{name}.parquet"
    part.to_parquet(path, index=False)
    print(f"{name:>6}: {len(part):>7,} rows  fraud {part['isFraud'].mean():.2%}  → {path}  ({os.path.getsize(path) / 1e6:.0f} MB)")

## Notes for tonight's commit

Write 3–5 lines in the repo's `notes/day01.md` in your own words:
- the fraud rate, and why accuracy is a useless score here
- what the weekly chart showed about fraud over time
- which columns looked suspicious in the leakage check, and what you decided
- the size of each split